# MLP 3x200 ReLU

Tres capas grandes. Es la variante de mayor capacidad para observar si sobreajusta o mejora en ventanas largas.

Entrena esta arquitectura para las 16 combinaciones de ventanas y guarda resultados en `data/mlp/mlp_3x200_relu.csv`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "util.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "util.py").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error

from util import (
    get_train_test,
    RANDOM_SEED,
    compare_to_benchmark,
    plot_benchmark_comparison,
    configure_mlflow,
    log_keras_grid_run,
)

keras.utils.set_random_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [2]:
MODEL_NAME = "mlp_3x200_relu"
EXPERIMENT_NAME = "MLP"
LOG_MODEL_ARTIFACT = False
ARCHITECTURE_PARAMS = {
    'hidden_layers': 3,
    'neurons': 200,
    'activation': 'relu',
    'regularization': 'none',
}
mlflow = configure_mlflow(EXPERIMENT_NAME)
RESULTS_PATH = PROJECT_ROOT / "data" / "mlp" / f"{MODEL_NAME}.csv"
HISTORY_PATH = PROJECT_ROOT / "data" / "mlp" / f"{MODEL_NAME}_history.csv"

input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

EPOCHS = 200
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
VALIDATION_SPLIT = 0.1

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS_PATH)
print("history:", HISTORY_PATH)


results: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/mlp/mlp_3x200_relu.csv
history: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/mlp/mlp_3x200_relu_history.csv


## Definicion de la red

In [3]:
def build_model(input_dim, output_dim):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(200, activation="relu"))
    model.add(Dense(200, activation="relu"))
    model.add(Dense(200, activation="relu"))
    model.add(Dense(output_dim))
    model.compile(loss="mean_absolute_error", optimizer=Adam(learning_rate=LEARNING_RATE))
    return model


## Entrenamiento en grid de ventanas

In [4]:
results = []
history_rows = []

for in_w in input_windows:
    for out_w in output_windows:
        d = get_train_test(input_window_size=in_w, output_window_size=out_w)

        X_train = d.X_train.reshape(d.X_train.shape[0], -1)
        X_test = d.X_test.reshape(d.X_test.shape[0], -1)

        keras.utils.set_random_seed(RANDOM_SEED)
        model = build_model(input_dim=X_train.shape[1], output_dim=d.y_train.shape[1])


        hist = model.fit(
            X_train,
            d.y_train,
            validation_split=VALIDATION_SPLIT,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0,
            shuffle=False,
        )

        y_pred_train = model.predict(X_train, verbose=0)
        y_pred_test = model.predict(X_test, verbose=0)

        row = {
            "model_name": MODEL_NAME,
            "input_window": in_w,
            "output_window": out_w,
            "MAE_train": mean_absolute_error(d.y_train, y_pred_train),
            "MAE_val": min(hist.history["val_loss"]),
            "MAE_test": mean_absolute_error(d.y_test, y_pred_test),
            "epochs": len(hist.history["loss"]),
            "n_params": model.count_params(),
        }
        results.append(row)

        for epoch, (loss, val_loss) in enumerate(
            zip(hist.history["loss"], hist.history["val_loss"]), start=1
        ):
            history_rows.append({
                "model_name": MODEL_NAME,
                "input_window": in_w,
                "output_window": out_w,
                "epoch": epoch,
                "loss": loss,
                "val_loss": val_loss,
            })

        run_name = f"{MODEL_NAME}_input{in_w}_output{out_w}"
        log_keras_grid_run(
            mlflow=mlflow,
            model=model,
            history=hist,
            run_name=run_name,
            model_name=MODEL_NAME,
            input_window=in_w,
            output_window=out_w,
            metrics_row=row,
            train_shape=X_train.shape,
            test_shape=X_test.shape,
            output_dim=d.y_train.shape[1],
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            validation_split=VALIDATION_SPLIT,
            extra_params=ARCHITECTURE_PARAMS,
            log_model_artifact=LOG_MODEL_ARTIFACT,
        )

        results_df = pd.DataFrame(results)
        history_df = pd.DataFrame(history_rows)
        results_df.to_csv(RESULTS_PATH, index=False)
        history_df.to_csv(HISTORY_PATH, index=False)

        print(
            f"input={in_w:>3} output={out_w:>3} | "
            f"train={row['MAE_train']:.6f} val={row['MAE_val']:.6f} "
            f"test={row['MAE_test']:.6f} epochs={row['epochs']} "
            f"params={row['n_params']}"
        )

results_df = pd.DataFrame(results)
history_df = pd.DataFrame(history_rows)
results_df


input=  5 output=  1 | train=0.009168 val=0.009034 test=0.014438 epochs=200 params=108223


input=  5 output=  5 | train=0.004377 val=0.004158 test=0.006466 epochs=200 params=108223


input=  5 output= 30 | train=0.001922 val=0.001757 test=0.002609 epochs=200 params=108223


input=  5 output= 90 | train=0.001328 val=0.001014 test=0.001574 epochs=200 params=108223


input= 10 output=  1 | train=0.008608 val=0.009034 test=0.014047 epochs=200 params=131223


...

input= 90 output=  1 | train=0.006331 val=0.009031 test=0.014129 epochs=200 params=499223


input= 90 output=  5 | train=0.003924 val=0.004171 test=0.006303 epochs=200 params=499223


input= 90 output= 30 | train=0.001943 val=0.001764 test=0.002541 epochs=200 params=499223


input= 90 output= 90 | train=0.001271 val=0.001005 test=0.001461 epochs=200 params=499223


,model_name,input_window,output_window,MAE_train,MAE_val,MAE_test,epochs,n_params
0,mlp_3x200_relu,5,1,0.009168,0.009034,0.014438,200,108223
1,mlp_3x200_relu,5,5,0.004377,0.004158,0.006466,200,108223
2,mlp_3x200_relu,5,30,0.001922,0.001757,0.002609,200,108223
3,mlp_3x200_relu,5,90,0.001328,0.001014,0.001574,200,108223
4,mlp_3x200_relu,10,1,0.008608,0.009034,0.014047,200,131223
5,mlp_3x200_relu,10,5,0.004131,0.004167,0.006313,200,131223
6,mlp_3x200_relu,10,30,0.001845,0.001765,0.002655,200,131223
7,mlp_3x200_relu,10,90,0.001243,0.001018,0.001506,200,131223
8,mlp_3x200_relu,30,1,0.007461,0.009025,0.014195,200,223223
9,mlp_3x200_relu,30,5,0.003966,0.004158,0.006509,200,223223


## Matrices de error

In [5]:
mae_train_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_train")
mae_val_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_val")
mae_test_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_test")

print("MAE train")
display(mae_train_matrix)
print("MAE val")
display(mae_val_matrix)
print("MAE test")
display(mae_test_matrix)


MAE train


output_window,1,5,30,90
input_window,,,,
5,0.009168,0.004377,0.001922,0.001328
10,0.008608,0.004131,0.001845,0.001243
30,0.007461,0.003966,0.001808,0.001243
90,0.006331,0.003924,0.001943,0.001271


MAE val


output_window,1,5,30,90
input_window,,,,
5,0.009034,0.004158,0.001757,0.001014
10,0.009034,0.004167,0.001765,0.001018
30,0.009025,0.004158,0.001758,0.001008
90,0.009031,0.004171,0.001764,0.001005


MAE test


output_window,1,5,30,90
input_window,,,,
5,0.014438,0.006466,0.002609,0.001574
10,0.014047,0.006313,0.002655,0.001506
30,0.014195,0.006509,0.002515,0.001373
90,0.014129,0.006303,0.002541,0.001461


## Heatmaps

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
matrices = [
    (mae_train_matrix, "MAE train"),
    (mae_val_matrix, "MAE val"),
    (mae_test_matrix, "MAE test"),
]
vmin = min(matrix.values.min() for matrix, _ in matrices)
vmax = max(matrix.values.max() for matrix, _ in matrices)

for ax, (matrix, title) in zip(axes, matrices):
    im = ax.imshow(matrix.values, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_xlabel("Ventana salida")
    ax.set_ylabel("Ventana entrada")
    ax.set_title(f"{title} - {MODEL_NAME}")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix.values[i, j]:.4f}", ha="center", va="center", color="white", fontsize=9)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_62659/2336802165.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Curvas de aprendizaje

In [7]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14), sharey=False)
axes = axes.ravel()

for ax, ((in_w, out_w), group) in zip(
    axes,
    history_df.groupby(["input_window", "output_window"], sort=True),
):
    ax.plot(group["epoch"], group["loss"], label="train")
    ax.plot(group["epoch"], group["val_loss"], label="val")
    ax.set_title(f"in={in_w}, out={out_w}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MAE")
    ax.grid(True, alpha=0.3)

axes[0].legend()
plt.suptitle(f"Curvas de aprendizaje - {MODEL_NAME}", y=1.02)
plt.tight_layout()
plt.show()


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_62659/1319896804.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Comparacion contra regresion lineal

In [8]:
comparison = compare_to_benchmark(results_df, benchmark="lr_benchmark")
display(comparison)

plot_benchmark_comparison(results_df, benchmark="lr_benchmark", model_name=MODEL_NAME)
plt.show()


,input_window,output_window,MAE_test,MAE_test_benchmark,delta,pct_delta
0,5,1,0.014438,0.012384,0.002054,16.584940
1,5,5,0.006466,0.005625,0.000841,14.956425
2,5,30,0.002609,0.002340,0.000269,11.497357
3,5,90,0.001574,0.001271,0.000303,23.801936
4,10,1,0.014047,0.012554,0.001492,11.888083
5,10,5,0.006313,0.005698,0.000615,10.793488
6,10,30,0.002655,0.002358,0.000297,12.572133
7,10,90,0.001506,0.001282,0.000224,17.439930
8,30,1,0.014195,0.012924,0.001271,9.832829
9,30,5,0.006509,0.005877,0.000633,10.764460


/var/folders/py/c5_xfbqn469g5_844mv32gt40000gn/T/ipykernel_62659/4288889828.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
